In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf chromadb sentence-transformers

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World",
    metadata={"source":"https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
#text data
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("data/python.txt", encoding="utf-8")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32420\3596417266.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
TextDocument = loader.load()

In [8]:
TextDocument

[Document(metadata={'source': 'data/python.txt'}, page_content='')]

In [9]:
# #PDF data
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("data/research.pdf")

# PDF_document = loader.load()

# PDF_document

## ingestion pipeline

In [10]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader, PyMuPDFLoader

### Documents

In [11]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyMuPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print(f"total docs = {num_docs}")
    print(f"total pages = {len(all_docs)}")
    return all_docs

In [12]:
all_pdf_documents = load_all_pdfs()

total docs = 34
total pages = 229


In [13]:
type(all_pdf_documents[5])

langchain_core.documents.base.Document

### Chunks

In [14]:
#chunks
# !pip install langchain_text_splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(document, chunk_size=500, chunk_overlap=50): # chunk_size means what is the maximuum num of charcter that each chunk is going to have
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(document)
    return chunked_docs

In [16]:
chunks = split_docs(all_pdf_documents)

In [17]:
len(chunks)

847

### Embedding

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
# os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxxx" #to avoid the warning: You are sending unauthenticated requests to the HF Hub.

In [20]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding dimensions", self.model.get_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding.....", embeddings.shape)
        return embeddings

In [21]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions 384


### Vector store

In [22]:
import chromadb
import uuid  #help us to create idx

In [23]:
class VectorStoreManager:

    def __init__(
        self,
        persist_direcctory="data/vector_store",
        collection_name="pdf_documents"
    ):
        self.persist_direcctory = persist_direcctory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):

        os.makedirs(
            self.persist_direcctory,
            exist_ok=True
        )

        # Create Chroma client
        self.client = chromadb.PersistentClient(
            path=self.persist_direcctory
        )

        # Create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for pdf embeddings in RAG",
                "hnsw:space": "cosine"
            }
        )

        print(
            "Initialized vector store:",
            self.collection_name
        )

        print(
            "Documents in collection:",
            self.collection.count()
        )

    def add_documents(self, documents, embeddings):

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents does not match number of embeddings"
            )

        # Lists that will be sent to Chroma
        ids = []
        documents_content = []
        embeddings_list = []
        all_metadata = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            doc_id = f"doc_{uuid.uuid4()}"

            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            all_metadata.append(metadata)

            documents_content.append(
                doc.page_content
            )

            embeddings_list.append(
                embedding.tolist()
            )
            
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print(
            "Total documents added:",
            len(documents_content)
        )

        print(
            "Documents in collection:",
            self.collection.count()
        )

In [24]:
vector_store = VectorStoreManager()

Initialized vector store: pdf_documents
Documents in collection: 847


In [25]:
# dara => documents => chunks => embeddings => store in vector store
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

embedding..... (847, 384)
Total documents added: 847
Documents in collection: 1694


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.4):
        #query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine similarity
        retrieved_docs = []
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "metadata": metadata,
                        "document": document,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"retrieve {len(retrieved_docs)} documents")
            
        else:
            print("no document found...")

        return retrieved_docs

In [28]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [29]:
rag_retriever.retrieve("What is git")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding..... (1, 384)
retrieve 5 documents


[{'metadata': {'keywords': '',
   'modDate': "D:20260601181536+00'00'",
   'title': 'git-cheat-sheet-education',
   'source': 'data/pdfs\\research8.pdf',
   'total_pages': 2,
   'moddate': '2026-06-01T18:15:36+00:00',
   'creationDate': "D:20260601181536+00'00'",
   'creationdate': '2026-06-01T18:15:36+00:00',
   'format': 'PDF 1.7',
   'content_length': 64,
   'trapped': '',
   'producer': 'pdfcpu v0.12.1 dev',
   'page': 0,
   'creator': 'Adobe Illustrator CC (Macintosh)',
   'subject': '',
   'author': '',
   'file_path': 'data/pdfs\\research8.pdf',
   'doc_index': 834},
  'document': 'Git for All Platforms\nhttp://git-scm.com\nnileshgale520@gmail.com',
  'distance': 0.381225049495697,
  'similarity_score': 0.618774950504303,
  'rank': 1},
 {'metadata': {'title': 'git-cheat-sheet-education',
   'producer': 'pdfcpu v0.12.1 dev',
   'creationdate': '2026-06-01T18:15:36+00:00',
   'source': 'data/pdfs\\research8.pdf',
   'page': 0,
   'content_length': 64,
   'subject': '',
   'modDate

### NVIDIA API KEY

In [30]:

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv() # to load api key from .env

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found. Check your .env file.")

llm = ChatOpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1",
    model="meta/llama-3.1-70b-instruct",
    temperature=0.1,
    max_tokens=1024
)

In [31]:
def generate_output(query, retriever, llm, top_k=3):

    results = retriever.retrieve(query, top_k)

    context = "\n\n".join(
        [doc["document"] for doc in results]
    ) if results else ""

    if not context:
        return "I could not find relevant information in the provided documents."

    prompt = f"""
        You are a retrieval-augmented question answering assistant.
        
        Answer the question using ONLY the provided context.
        
        Rules:
        1. Do not use outside knowledge.
        2. Do not invent or assume information.
        3. If the answer is not present in the context, say:
           "I could not find the answer in the provided documents."
        4. Give a concise and direct answer.
        
        Context:
        {context}
        
        Question:
        {query}
        
        Answer:
        """

    response = llm.invoke(prompt)
    return response.content

In [32]:
answer = generate_output(
    "What is Logistic regression",
    rag_retriever,
    llm
)

print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding..... (1, 384)
retrieve 3 documents
Logistic Regression is a supervised ML algorithm for classification problems that predicts the probability that an input belongs to a specific class or category.


### OPENAI API

In [33]:
# API_KEY_OPENAI = ""

In [34]:
# !pip install -U langchain-openai

# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(
#     openai_api_key = API_KEY_OPENAI,
#     model="gpt-5.4",
#     temperature=0.1,
#     max_token=1024
# )

In [35]:
# def generate_output(query, retriever, llm, top_k=3):
#     results = retriever.retrieve(query, top_k)

#     context = "\n".join([doc["document"] for doc in results]) if results else ""

#     if not context:
#         print("we found no relevant context for the given query")

#     #context + query
#     prompt = f""" use given context to generate the answer for the query
#                 Context: {context}
#                 Query : {query}"""

#     response = llm.invoke(prompt) #expecting a string as prompt

#     return response.content

In [36]:
# answer = generate_output("What is RAG", rag_retriever, llm)
# print(answer)

### GROQ

In [37]:
# API_KEY_GROQ = ""

In [38]:
# !pip install langchain-groq

In [39]:
# from langchain_groq import ChatGroq

# llm = ChatOpenAI(
#      groq_api_key = API_KEY_GROQ,
#      model="qwen/qwen3-32b",
#      temperature=0.1,
#      max_token=1024
# )

In [40]:
# def generate_output(query, retriever, llm, top_k=3):
#     results = retriever.retrieve(query, top_k)

#     context = "\n".join([doc["document"] for doc in results]) if results else ""

#     if not context:
#         print("we found no relevant context for the given query")

#     #context + query
#     prompt = f""" use given context to generate the answer for the query
#                 Context: {context}
#                 Query : {query}"""

#     response = llm.invoke([prompt,format(context=context, query=query)]) #expecting a string as prompt

#     return response.content

In [41]:
# answer = generate_output("What is RAG", rag_retriever, llm)
# print(answer)

## RAG Pipeline Evaluation

Measures two key metrics:
- **Retrieval Hit Rate** - did the retriever fetch the right chunk for each question?
- **Groundedness Rate** - did the LLM hallucinate? (LLM-as-judge)


In [42]:
import pandas as pd
from tqdm import tqdm

# Test set: 20 questions grounded in actual PDFs in data/pdfs/
# must_contain = keyword guaranteed to appear in the correct source chunk
TEST_SET = [
    # RAG Survey (research1.pdf)
    {"question": "What are the main challenges faced by Large Language Models that RAG addresses?",
     "must_contain": "hallucination"},
    {"question": "What does RAG stand for and what is its main purpose?",
     "must_contain": "retrieval-augmented generation"},
    # Evaluation Metrics for Classification (research4.pdf)
    {"question": "What is a Confusion Matrix used for?",
     "must_contain": "confusion matrix"},
    {"question": "What does False Positive mean in a confusion matrix?",
     "must_contain": "type i error"},
    {"question": "What is a True Negative in classification evaluation?",
     "must_contain": "true negative"},
    # Evaluation Metrics for Regression (research5.pdf)
    {"question": "What is Mean Squared Error (MSE) and how is it calculated?",
     "must_contain": "mean squared error"},
    {"question": "Why are differences squared in the MSE formula?",
     "must_contain": "squared"},
    # Logistic Regression (research20.pdf)
    {"question": "What function does Logistic Regression use to map values to a probability range?",
     "must_contain": "sigmoid"},
    {"question": "Is Logistic Regression used for regression or classification problems?",
     "must_contain": "classification"},
    # CNN / Deep Learning (research3.pdf)
    {"question": "What type of neural network is used for binary image classification of cats and dogs?",
     "must_contain": "convolutional neural network"},
    {"question": "What dataset is used for the cats vs dogs image classification project?",
     "must_contain": "dogs vs cats"},
    # Supervised ML Assignment 1 - House Pricing (research14.pdf)
    {"question": "What ML model is used to predict house prices in the HomeVista Properties assignment?",
     "must_contain": "linear regression"},
    {"question": "What company wants to automate house pricing using Machine Learning?",
     "must_contain": "homevista"},
    # Supervised ML Assignment 2 - Employee Attrition (research10.pdf)
    {"question": "What is the goal of the TalentCore Pvt. Ltd. ML assignment?",
     "must_contain": "leave the company"},
    {"question": "Which regularization techniques are compared in the employee attrition assignment?",
     "must_contain": "l1"},
    # Web Scraping / HTML (research16.pdf)
    {"question": "What does HTML stand for and what is it used for?",
     "must_contain": "hypertext markup language"},
    # Git and GitHub (research7.pdf / research8.pdf)
    {"question": "What is the purpose of a .gitignore file?",
     "must_contain": "gitignore"},
    # Reinforcement Learning - Mario (research23.pdf)
    {"question": "What RL algorithm is used in the Mario-playing agent assignment?",
     "must_contain": "double deep q"},
    # Cross-topic (RAG Survey)
    {"question": "How does RAG overcome the problem of outdated knowledge in LLMs?",
     "must_contain": "retrieval"},
    {"question": "What are the components of a RAG system?",
     "must_contain": "retrieval"},
]

### Metric 1 - Retrieval Hit Rate

Checks if the retrieved chunks contain the keyword we expect for each question.
Tells us whether the vector search is finding the right documents.


In [43]:
def retrieval_hit(retrieved_docs, must_contain: str) -> bool:
    """Return True if must_contain phrase appears in any retrieved chunk."""
    combined = " ".join(d["document"].lower() for d in retrieved_docs)
    return must_contain.lower() in combined


### Metric 2 - Groundedness Rate (LLM-as-Judge)

Sends the retrieved context + model answer back to the LLM and asks:
"Is every claim in the answer supported by the context? Reply 1 (yes) or 0 (no)."

A score of 0 means the model hallucinated information not in the retrieved documents.


In [44]:
def llm_judge_groundedness(question, context, answer, llm) -> int:
    """Ask the LLM to judge whether the answer is grounded in the context.
    Returns 1 (grounded / no hallucination) or 0 (hallucinated).
    """
    judge_prompt = (
        "You are a strict evaluator. Given a CONTEXT, a QUESTION, and an ANSWER,\n"
        "respond with ONLY the single character \"1\" if the answer is fully supported by the context\n"
        "(no hallucinated facts), or \"0\" if the answer includes any claim not present in the context.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{question}\n\n"
        f"ANSWER:\n{answer}\n\n"
        "Respond with only 1 or 0, nothing else."
    )
    verdict = llm.invoke(judge_prompt).content.strip()
    return 1 if verdict.startswith("1") else 0


### Run Evaluation Loop

Make sure all pipeline cells above are already run
so that `rag_retriever` and `llm` are defined.


In [45]:
TOP_K_EVAL     = 5
THRESHOLD_EVAL = 0.4

eval_results   = []
hit_count      = 0
grounded_count = 0

for item in tqdm(TEST_SET, desc="Evaluating"):
    question     = item["question"]
    must_contain = item["must_contain"]

    # Step 1: Retrieve relevant chunks
    retrieved_docs = rag_retriever.retrieve(
        question, top_k=TOP_K_EVAL, score_threshold=THRESHOLD_EVAL
    )
    context = "\n\n".join(d["document"] for d in retrieved_docs) if retrieved_docs else ""

    # Step 2: Generate answer
    answer = generate_output(question, rag_retriever, llm, top_k=TOP_K_EVAL)

    # Step 3: Score both metrics
    hit      = retrieval_hit(retrieved_docs, must_contain) if retrieved_docs else False
    grounded = llm_judge_groundedness(question, context, answer, llm) if context else 0

    hit_count      += int(hit)
    grounded_count += grounded

    eval_results.append({
        "question"            : question,
        "answer"              : answer,
        "num_chunks_retrieved": len(retrieved_docs),
        "retrieval_hit"       : hit,
        "grounded"            : bool(grounded),
    })

n = len(TEST_SET)
print("\n" + "=" * 50)
print("  RAG Evaluation Results")
print("=" * 50)
print(f"  Retrieval Hit Rate : {hit_count}/{n} = {hit_count/n:.1%}")
print(f"  Groundedness Rate  : {grounded_count}/{n} = {grounded_count/n:.1%}")
print("=" * 50)


Evaluating:   0%|                                                                               | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding..... (1, 384)
retrieve 5 documents


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding..... (1, 384)
retrieve 5 documents


...

embedding..... (1, 384)
retrieve 5 documents


Evaluating:  60%|██████████████████████████████████████████                            | 12/20 [10:07<05:46, 43.29s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:  60%|██████████████████████████████████████████                            | 12/20 [10:08<06:45, 50.70s/it]


AcceleratorError: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Save Results to CSV


In [46]:
df_eval = pd.DataFrame(eval_results)
df_eval.to_csv("rag_eval_results.csv", index=False)
print("Saved to rag_eval_results.csv")
df_eval


Saved to rag_eval_results.csv


,question,answer,num_chunks_retrieved,retrieval_hit,grounded
0,What are the main challenges faced by Large La...,I could not find the answer in the provided do...,5,False,False
1,What does RAG stand for and what is its main p...,RAG stands for Retrieval-Augmented. The main p...,5,False,False
2,What is a Confusion Matrix used for?,A Confusion Matrix is used to evaluate the per...,2,True,True
3,What does False Positive mean in a confusion m...,"In a Confusion Matrix, False Positive (FP) mea...",4,True,True
4,What is a True Negative in classification eval...,A True Negative (TN) in classification evaluat...,5,True,True
5,What is Mean Squared Error (MSE) and how is it...,Mean Squared Error (MSE) is an evaluation metr...,5,True,True
6,Why are differences squared in the MSE formula?,Differences are squared in the MSE formula to ...,5,True,True
7,What function does Logistic Regression use to ...,Sigmoid function.,4,True,True
8,Is Logistic Regression used for regression or ...,Logistic Regression is used for classification...,5,True,True
9,What type of neural network is used for binary...,Convolutional Neural Network (CNN),5,True,True
